<a href="https://colab.research.google.com/github/kuunalmistry/Google_Colab_Clg_Experiments_Projects/blob/main/Brain_tumor_classification_KuunalMistry_2410130.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install gradio tensorflow keras numpy pillow scikit-learn matplotlib

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
from google.colab import drive
import os
import numpy as np
from PIL import Image
import gradio as gr
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Flatten, Conv2D, MaxPooling2D, BatchNormalization
from sklearn.preprocessing import OneHotEncoder
import matplotlib.pyplot as plt

In [4]:
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
data = []
paths = []
labels = []
root_path = '/content/drive/MyDrive/Brain_Tumor_Dataset'

In [6]:
for label, folder in enumerate(['no', 'yes']):
    folder_path = os.path.join(root_path, folder)
    for file in os.listdir(folder_path):
        if '.jpg' in file:
            img_path = os.path.join(folder_path, file)
            paths.append(img_path)
            labels.append(label)
            img = Image.open(img_path)
            img = img.resize((128, 128))
            img_array = np.array(img)
            if img_array.shape == (128, 128, 3):
                data.append(img_array)

In [7]:
data = np.array(data)
labels = np.array(labels).reshape(-1, 1)

In [8]:
enc = OneHotEncoder()
enc.fit([[0], [1]])
labels = enc.transform(labels).toarray()

In [9]:
model = Sequential([
    Conv2D(32, (3, 3), activation='relu', input_shape=(128, 128, 3)),
    BatchNormalization(),
    MaxPooling2D((2, 2)),
    Conv2D(64, (3, 3), activation='relu'),
    BatchNormalization(),
    MaxPooling2D((2, 2)),
    Conv2D(128, (3, 3), activation='relu'),
    BatchNormalization(),
    MaxPooling2D((2, 2)),
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(2, activation='softmax')
])

/usr/local/lib/python3.13/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [10]:
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

In [11]:
history = model.fit(data, labels, epochs=10, batch_size=32, validation_split=0.2, verbose=1)

Epoch 1/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 11s 2s/step - accuracy: 0.8175 - loss: 1.7767 - val_accuracy: 0.0000e+00 - val_loss: 236.9075
Epoch 2/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 9s 2s/step - accuracy: 0.9635 - loss: 0.3031 - val_accuracy: 0.0000e+00 - val_loss: 228.5194
Epoch 3/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 10s 2s/step - accuracy: 0.9927 - loss: 0.0295 - val_accuracy: 0.0000e+00 - val_loss: 172.9622
Epoch 4/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 8s 1s/step - accuracy: 0.9927 - loss: 0.0990 - val_accuracy: 0.0000e+00 - val_loss: 148.3916
Epoch 5/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 9s 2s/step - accuracy: 0.9927 - loss: 0.0271 - val_accuracy: 0.0000e+00 - val_loss: 125.6832
Epoch 6/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 8s 1s/step - accuracy: 1.0000 - loss: 1.1648e-05 - val_accuracy: 0.0000e+00 - val_loss: 97.2466
Epoch 7/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 11s 1s/step - accuracy: 0.9927 - loss: 0.0385 - val_accuracy: 0.0000e+00 - val_loss: 100.4669
Epoch 8/10
5/5 ━━━━━━━━━━━━━━━━━━━━ 8s 2s/step - accuracy: 0.9927 - loss: 0.0761 - val_accu

In [12]:
def predict_brain_tumor(img):

    img = img.resize((128, 128))
    img_array = np.array(img).reshape(-1, 128, 128, 3)


    prediction = model.predict(img_array)
    classification = np.where(prediction == np.amax(prediction))[1][0]
    confidence = prediction[0][classification] * 100


    label = 'Tumor' if classification == 1 else 'Normal'
    result = f"{round(confidence, 2)}% Confidence\nThis is {label}"
    return result


In [13]:
interface = gr.Interface(
    fn=predict_brain_tumor,
    inputs=gr.Image(type="pil"),
    outputs=gr.Textbox(),
    title="Brain Tumor Classification",
    description="Upload an MRI image to classify if it contains a brain tumor."
)

interface.launch(share=False)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Note: opening Chrome Inspector may crash demo inside Colab notebooks.
* To create a public link, set `share=True` in `launch()`.


<IPython.core.display.Javascript object>